<a href="https://colab.research.google.com/github/kjunseo001-glitch/prompt-eng-interactive-tutorial/blob/master/Yfinance_%EA%B8%81%EC%96%B4%EC%98%A4%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 셀 X - Claude 직접 분석용 데이터 수집 v2
# TICKER만 바꾸고 실행 → JSON 복사 → Claude에게 붙여넣기

import yfinance as yf, json, numpy as np, pandas as pd

TICKER = "NVDA"  # ← 여기만 바꾸면 됨

t    = yf.Ticker(TICKER)
info = t.info or {}
g    = lambda k, d=None: info.get(k, d)

# ── 기본 재무 ─────────────────────────────────────────────────────
price  = g("currentPrice") or g("regularMarketPrice")
mktcap = g("marketCap"); shares = g("sharesOutstanding")
rev_ttm= g("totalRevenue"); ni_ttm = g("netIncomeToCommon")
ocf    = g("operatingCashflow"); capex = g("capitalExpenditures")
fcf    = g("freeCashflow"); cash = g("totalCash"); debt = g("totalDebt")
bps    = g("bookValue"); roe = g("returnOnEquity"); roa = g("returnOnAssets")
gm     = g("grossMargins"); opm = g("operatingMargins"); npm = g("profitMargins")
per    = g("trailingPE"); pbr = g("priceToBook"); eps = g("trailingEps")
hi52   = g("fiftyTwoWeekHigh"); lo52 = g("fiftyTwoWeekLow")
psr    = round(mktcap/rev_ttm, 2) if mktcap and rev_ttm else None
stage  = "Stage1" if (ni_ttm or 0) <= 0 else "Stage2"

# Runway (Stage1만)
burn   = abs(ocf or 0) + abs(capex or 0)
runway = round(cash / burn * 12, 1) if burn > 0 and stage == "Stage1" else None

# ── 10년물 금리 (금리 보정용) ─────────────────────────────────────
rate_10y = None
try:
    tnx = yf.Ticker("^TNX")
    rate_10y = round(float(tnx.history(period="5d")["Close"].iloc[-1]) / 100, 4)
except:
    pass

# ── 성장률 & 분기 실적 (YoY TTM + GM 추세) ───────────────────────
rev_growth_yoy = None
quarterly_gm   = []
eps_fwd        = g("forwardEps")
eps_growth     = round((eps_fwd / eps - 1) * 100, 1) if eps_fwd and eps and eps > 0 else None

try:
    qf = t.quarterly_financials
    if qf is not None and not qf.empty:
        rev_row = qf.loc["Total Revenue"] if "Total Revenue" in qf.index else None
        gp_row  = qf.loc["Gross Profit"]  if "Gross Profit"  in qf.index else None
        if rev_row is not None and len(rev_row) >= 8:
            r_ttm  = rev_row.iloc[:4].sum()
            r_prev = rev_row.iloc[4:8].sum()
            rev_growth_yoy = round((r_ttm / r_prev - 1) * 100, 1) if r_prev else None
        if gp_row is not None and rev_row is not None:
            for i in range(min(4, len(rev_row))):
                gm_q = round(float(gp_row.iloc[i]) / float(rev_row.iloc[i]) * 100, 1) if float(rev_row.iloc[i]) != 0 else None
                quarterly_gm.append({"quarter": str(qf.columns[i].date()), "gm_pct": gm_q})
except:
    pass

# ── Beat/Miss 이력 (최근 8분기) ───────────────────────────────────
beat_miss          = []
next_earnings_date = None
try:
    ed = t.earnings_dates
    if ed is not None and not ed.empty:
        ed_clean = ed.dropna(subset=["EPS Estimate", "Reported EPS"])
        for idx, row in ed_clean.head(8).iterrows():
            est, rep = row["EPS Estimate"], row["Reported EPS"]
            surprise = round((rep - est) / abs(est) * 100, 1) if est != 0 else None
            beat_miss.append({
                "date":         str(idx.date()),
                "estimate":     round(float(est), 2),
                "actual":       round(float(rep), 2),
                "surprise_pct": surprise,
                "beat":         bool(rep > est),
            })
        future = ed[ed.index > pd.Timestamp.now(tz="UTC")]
        if not future.empty:
            next_earnings_date = str(future.index[-1].date())
except:
    pass

# ── 애널리스트 ────────────────────────────────────────────────────
analyst = {
    "target_mean":   g("targetMeanPrice"), "target_high": g("targetHighPrice"),
    "target_low":    g("targetLowPrice"),  "recommendation": g("recommendationKey"),
    "num_analysts":  g("numberOfAnalystOpinions"),
}

# ── 기관/수급 ─────────────────────────────────────────────────────
short_pct = g("shortPercentOfFloat")
inst_pct  = g("institutionPercentHeld")
float_sh  = g("floatShares")

# ── 내부자 ───────────────────────────────────────────────────────
insider = {}
try:
    trans = t.insider_transactions
    if trans is not None and not trans.empty:
        trans = trans.copy()
        trans["Value"]  = pd.to_numeric(trans.get("Value",  0), errors="coerce").fillna(0)
        trans["Shares"] = pd.to_numeric(trans.get("Shares", 0), errors="coerce").fillna(0)
        buys  = trans[trans["Transaction"].str.contains("Buy|Purchase", case=False, na=False)]
        sells = trans[trans["Transaction"].str.contains("Sell|Sale",    case=False, na=False)]
        # CEO + CFO 모두 포함
        ceo_cfo_b = buys[buys["Insider"].str.contains(
            "CEO|CFO|Chief Executive|Chief Financial", case=False, na=False)]
        buy_v  = float(buys["Value"].sum()); sell_v = float(sells["Value"].sum())
        insider = {
            "buy_value_3m":      round(buy_v, 0),
            "sell_value_3m":     round(sell_v, 0),
            "net_buy_value":     round(buy_v - sell_v, 0),
            "ceo_cfo_bought":    len(ceo_cfo_b) > 0,
            "large_single_sell": float(sells["Value"].max()) > sell_v * 0.5 if not sells.empty else False,
        }
except:
    pass

# ── 기술적 지표 (5년 일봉) ────────────────────────────────────────
hist = t.history(period="5y", interval="1d")
if isinstance(hist.columns, pd.MultiIndex): hist.columns = hist.columns.get_level_values(0)

cl, hi, lo, vol = hist["Close"], hist["High"], hist["Low"], hist["Volume"]

def ema(s, n):
    return s.ewm(span=n, adjust=False).mean()

def rsi_calc(s, p=14):
    d  = s.diff()
    g_ = d.clip(lower=0).ewm(alpha=1/p, adjust=False).mean()
    l_ = (-d.clip(upper=0)).ewm(alpha=1/p, adjust=False).mean()
    return 100 - 100 / (1 + g_ / l_.replace(0, np.nan))

def stoch(hi, lo, cl, k=5, sm=3, d=3):
    ll, hh = lo.rolling(k).min(), hi.rolling(k).max()
    raw = 100 * (cl - ll) / (hh - ll).replace(0, np.nan)
    sk  = raw.rolling(sm).mean()
    return sk, sk.rolling(d).mean()

def atr_calc(hi, lo, cl, p=14):
    tr = pd.concat([hi - lo, (hi - cl.shift()).abs(), (lo - cl.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1/p, adjust=False).mean()

e5, e20, e50, e200 = ema(cl,5), ema(cl,20), ema(cl,50), ema(cl,200)
wk = hist.resample("W").agg({"Close":"last","High":"max","Low":"min","Volume":"sum"})
we5, we20, we60 = ema(wk["Close"],5), ema(wk["Close"],20), ema(wk["Close"],60)
rsi14           = rsi_calc(cl)
sk_f, sd_f      = stoch(hi, lo, cl, 5,  3, 3)
sk_s, sd_s      = stoch(hi, lo, cl, 10, 6, 6)
atr14           = atr_calc(hi, lo, cl)
vol20           = vol.rolling(20).mean()
hi52w, lo52w    = cl.rolling(252).max(), cl.rolling(252).min()
vwap30          = (cl * vol).rolling(30).sum() / vol.rolling(30).sum()

def fv(v):
    try: return round(float(v), 2) if not np.isnan(float(v)) else None
    except: return None

cur       = float(cl.iloc[-1])
atr_pct   = round(fv(atr14.iloc[-1]) / cur * 100, 2) if fv(atr14.iloc[-1]) and cur else None
vol_group = ("고변동성(>5%)"  if atr_pct and atr_pct > 5  else
             "중변동성(2~5%)" if atr_pct and atr_pct >= 2 else "저변동성(<2%)")

# 최근 5일 OHLCV
recent = [
    {"date": str(hist.index[i].date()), "open": round(float(hist["Open"].iloc[i]),2),
     "high": round(float(hi.iloc[i]),2), "low": round(float(lo.iloc[i]),2),
     "close": round(float(cl.iloc[i]),2), "volume": int(vol.iloc[i])}
    for i in range(-5, 0)
]

# 피벗 포인트
pivots, w = [], 10
for i in range(w, len(cl) - w):
    seg = cl.iloc[i-w:i+w+1]
    if cl.iloc[i] == seg.max():
        pivots.append({"type":"high","price":round(float(cl.iloc[i]),2),"date":str(cl.index[i].date())})
    elif cl.iloc[i] == seg.min():
        pivots.append({"type":"low", "price":round(float(cl.iloc[i]),2),"date":str(cl.index[i].date())})
pivots = pivots[-8:]

# 크로스 시그널
crosses = []
for i in range(-20, 0):
    pd20 = float(e20.iloc[i-1]) - float(e50.iloc[i-1])
    cd20 = float(e20.iloc[i])   - float(e50.iloc[i])
    if pd20 < 0 < cd20:   crosses.append({"type":"golden","date":str(cl.index[i].date())})
    elif pd20 > 0 > cd20: crosses.append({"type":"dead",   "date":str(cl.index[i].date())})

# ── 최종 출력 ─────────────────────────────────────────────────────
result = {
    "ticker": TICKER,
    "fundamental": {
        "price": price, "mktcap": mktcap, "shares": shares,
        "high52": hi52, "low52": lo52,
        "rev_ttm": rev_ttm, "ni_ttm": ni_ttm,
        "ocf": ocf, "fcf": fcf, "cash": cash, "debt": debt,
        "gross_margin": round(gm*100,1)  if gm  else None,
        "op_margin":    round(opm*100,1) if opm else None,
        "net_margin":   round(npm*100,1) if npm else None,
        "roe": round(roe*100,1) if roe else None,
        "roa": round(roa*100,1) if roa else None,
        "per": per, "psr": psr, "pbr": pbr, "eps": eps, "bps": bps,
        "eps_fwd":        eps_fwd,
        "eps_growth_pct": eps_growth,
        "rev_growth_yoy": rev_growth_yoy,
        "quarterly_gm":   quarterly_gm,    # 최근 4분기 GM 추세
        "rate_10y":       rate_10y,         # 금리 보정용
        "stage": stage, "runway_months": runway,
        "float_shares": float_sh,
        "analyst": analyst,
        "short_pct_float":   round(float(short_pct)*100, 2) if short_pct else None,
        "institutional_pct": round(float(inst_pct)*100,  2) if inst_pct  else None,
        "insider": insider,
    },
    "earnings": {
        "next_date": next_earnings_date,
        "beat_miss": beat_miss,             # 최근 8분기 Beat/Miss
    },
    "technical": {
        "current_price": fv(cl.iloc[-1]),
        "atr_14":        fv(atr14.iloc[-1]),
        "atr_pct":       atr_pct,
        "vol_group":     vol_group,
        "vwap_30":       fv(vwap30.iloc[-1]),
        "recent_ohlcv":  recent,
        "ema": {
            "e5": fv(e5.iloc[-1]), "e20": fv(e20.iloc[-1]),
            "e50": fv(e50.iloc[-1]), "e200": fv(e200.iloc[-1]),
            "weekly": {"e5w": fv(we5.iloc[-1]), "e20w": fv(we20.iloc[-1]), "e60w": fv(we60.iloc[-1])}
        },
        "ema_alignment": {
            "daily":  ("정배열" if fv(e5.iloc[-1])>fv(e20.iloc[-1])>fv(e50.iloc[-1])>fv(e200.iloc[-1]) else
                       "역배열" if fv(e5.iloc[-1])<fv(e20.iloc[-1])<fv(e50.iloc[-1])<fv(e200.iloc[-1]) else "혼합"),
            "weekly": ("정배열" if fv(we5.iloc[-1])>fv(we20.iloc[-1])>fv(we60.iloc[-1]) else
                       "역배열" if fv(we5.iloc[-1])<fv(we20.iloc[-1])<fv(we60.iloc[-1]) else "혼합"),
        },
        "rsi_14": fv(rsi14.iloc[-1]),
        "stochastic": {
            "fast_k": fv(sk_f.iloc[-1]), "fast_d": fv(sd_f.iloc[-1]),
            "slow_k": fv(sk_s.iloc[-1]), "slow_d": fv(sd_s.iloc[-1]),
        },
        "volume": {
            "current": int(vol.iloc[-1]), "avg20": int(vol20.iloc[-1]),
            "ratio": round(float(vol.iloc[-1] / vol20.iloc[-1]), 2),
        },
        "range_52w": {
            "high": fv(hi52w.iloc[-1]), "low": fv(lo52w.iloc[-1]),
            "pct_from_high": round((cur / float(hi52w.iloc[-1]) - 1) * 100, 2),
            "pct_from_low":  round((cur / float(lo52w.iloc[-1]) - 1) * 100, 2),
        },
        "price_vs_ema": {
            "vs_e200": round((cur / float(e200.iloc[-1]) - 1) * 100, 2),
            "vs_e50":  round((cur / float(e50.iloc[-1])  - 1) * 100, 2),
        },
        "cross_signals": crosses,
        "pivots": pivots,
    }
}

print(json.dumps(result, ensure_ascii=False, indent=2))
print(f"\n✅ {TICKER} 수집 완료 — 위 JSON을 전체 복사해서 Claude에게 붙여넣기")
print(f"   Stage: {stage} | ATR%: {atr_pct}% ({vol_group})")
if rate_10y:        print(f"   10년물 금리: {rate_10y*100:.2f}%")
if rev_growth_yoy:  print(f"   매출 YoY 성장률: {rev_growth_yoy}% | EPS 성장률: {eps_growth}%")
if next_earnings_date: print(f"   다음 실적발표: {next_earnings_date}")
if beat_miss:
    beat_cnt = sum(1 for x in beat_miss if x["beat"])
    print(f"   Beat/Miss: {beat_cnt}/{len(beat_miss)} Beat ({len(beat_miss)}분기)")
if insider:         print(f"   내부자 순매수: ${insider.get('net_buy_value',0):,.0f} | CEO/CFO매수: {insider.get('ceo_cfo_bought')}")
if short_pct:       print(f"   공매도: {round(float(short_pct)*100,1)}% | 기관지분율: {round(float(inst_pct)*100,1) if inst_pct else 'N/A'}%")